In [2]:
import pandas as pd
import numpy as np
import fastf1
import requests

### Function to get Best FP1 Time

In [3]:
def get_fp1_best(FP1):

    laps = FP1.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

### Function to get Best FP2 Time

In [4]:
def get_fp2_best(FP2):

    laps = FP2.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

### Function to get Best FP3 Time

In [5]:
def get_fp3_best(FP3):

    laps = FP3.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

In [6]:
def get_sector_times (FP2):

    laps = FP2.laps

    sector_times = laps.groupby("Driver")[[
    "Sector1Time",
    "Sector2Time",
    "Sector3Time"
    ]].mean().apply(lambda x: x.dt.total_seconds()).reset_index()

    return sector_times

In [7]:
def get_average_laptime (FP2):

    laps = FP2.laps

    avg_lap = laps.groupby("Driver")["LapTime"].mean().dt.total_seconds().reset_index()

    return avg_lap

In [8]:
def get_stint(FP2):

    laps = FP2.laps

    stints = (
        laps.groupby(["Driver","Stint"])
        .size()
        .reset_index(name="LapCount")
    )

    longest_stint = (
        stints.groupby("Driver")["LapCount"]
        .max()
        .reset_index(name="LongestStint")
    )

    return longest_stint

In [9]:
def get_tyre_compound (FP2):

    laps = FP2.laps

    tyres = laps.groupby("Driver")["Compound"].agg(lambda x: x.mode()[0]).reset_index()

    return tyres

In [10]:
def get_qualifying_data(Qualify):

    results = Qualify.results.copy()

    qual_time = results[["Q3", "Q2", "Q1"]].bfill(axis=1).iloc[:, 0]

    qual = results[["Abbreviation"]].copy()
    qual["Qualifying_Time(s)"] = qual_time.dt.total_seconds()

    qual.rename(columns={"Abbreviation": "Driver"}, inplace=True)

    return qual

In [11]:
def get_starting_position(Qualify):

    quali_results = Qualify.results[['Abbreviation', 'Position']]
    quali_results.rename(columns={"Abbreviation":"Driver"},inplace=True)

    return quali_results

In [12]:
def get_race_results(Result):

    race = Result.results[["Abbreviation","Position"]]
    race.rename(columns={"Abbreviation":"Driver"},inplace=True)

    return race

In [13]:
def Rain(API_KEY,lat,lon,forecast_time):
    url = f"http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"

    response = requests.get(url)
    data = response.json()

    if "list" not in data:
        print("API ERROR:", data)
    else:

        forecast_data = next(
            (f for f in data["list"] if f["dt_txt"] == forecast_time),
            None
        )
        if forecast_data:
            rain_probability = forecast_data.get("pop", 0)
            temperature = forecast_data["main"]["temp"]
        else:
            forecast_data = data["list"][0]
            rain_probability = forecast_data.get("pop", 0)
            temperature = forecast_data["main"]["temp"]

    return rain_probability,temperature

# Australia GP Data(For Training)

In [13]:
season = 2026
race = 'Australia'

FP1 = fastf1.get_session(season, race, 'FP1')
FP1.load()
FP2 = fastf1.get_session(season, race, 'FP2')
FP2.load()
FP3 = fastf1.get_session(season, race, 'FP3')
FP3.load()

Qualify = fastf1.get_session(season, race, 'Q')
Qualify.load()

Result = fastf1.get_session(season, race, 'R')
Result.load()

print(f"Done Loading Season {season} & Race {race}")

fp1_best = get_fp1_best(FP1)
fp2_best = get_fp2_best(FP2)
fp3_best = get_fp3_best(FP3)
sector_time_best = get_sector_times(FP2)
average_laptime = get_average_laptime(FP2)
stint = get_stint(FP2)
tyre = get_tyre_compound(FP2)
qualifying_time = get_qualifying_data(Qualify)
starting_position = get_starting_position(Qualify)
result = get_race_results(Result)

print("Data Extraction Done.")

fp1_best.set_index("Driver",inplace=True)
fp2_best.set_index("Driver",inplace=True)
fp3_best.set_index("Driver",inplace=True)
sector_time_best.set_index("Driver",inplace=True)
average_laptime.set_index("Driver",inplace=True)
stint.set_index("Driver",inplace=True)
tyre.set_index("Driver",inplace=True)
qualifying_time.set_index("Driver",inplace=True)
starting_position.set_index("Driver",inplace=True)
result.set_index("Driver",inplace=True)

print("Setting Index Done.")

df = fp1_best.copy()
df.rename(columns={"LapTime":"FP1_BestTime(s)"},inplace=True)
df[["FP2_BestTime(s)"]] = fp2_best[["LapTime"]]
df[["FP3_BestTime(s)"]] = fp3_best[["LapTime"]]
df[["Sector1Time(s)","Sector2Time(s)","Sector3Time(s)"]] = sector_time_best[["Sector1Time","Sector2Time","Sector3Time"]]
df[["Average_Laptime(s)"]] = average_laptime[["LapTime"]]
df[["Longest_Stint"]] = stint[["LongestStint"]]
df[["Tyre_Compound"]] = tyre[["Compound"]]
df[["Qualifying_Time(s)"]] = qualifying_time[["Qualifying_Time(s)"]]
df[["Starting_Pos"]] = starting_position[["Position"]]
df[["Race_Result"]] = result[["Position"]]

print("Data Transformation.")

fp1_best.reset_index(inplace=True)
fp2_best.reset_index(inplace=True)
fp3_best.reset_index(inplace=True)
sector_time_best.reset_index(inplace=True)
average_laptime.reset_index(inplace=True)
stint.reset_index(inplace=True)
tyre.reset_index(inplace=True)
qualifying_time.reset_index(inplace=True)
starting_position.reset_index(inplace=True)
result.reset_index(inplace=True)
df.reset_index(inplace=True)

print("Resetting Index Done.")


req         WARNING 	DEFAULT CACHE ENABLED! (12.29 GB) /home/satyam/.cache/fastf1


core           INFO 	Loading data for Australian Grand Prix - Practice 1 [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 14
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 14)
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '5', '6', '10', '11', '12', '14', '16', '18', '23', '27', '30',

Done Loading Season 2026 & Race Australia
Data Extraction Done.
Setting Index Done.
Data Transformation.
Resetting Index Done.


/tmp/ipykernel_13829/2861812837.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  quali_results.rename(columns={"Abbreviation":"Driver"},inplace=True)
/tmp/ipykernel_13829/2097842272.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race.rename(columns={"Abbreviation":"Driver"},inplace=True)


In [14]:
df

,Driver,FP1_BestTime(s),FP2_BestTime(s),FP3_BestTime(s),Sector1Time(s),Sector2Time(s),Sector3Time(s),Average_Laptime(s),Longest_Stint,Tyre_Compound,Qualifying_Time(s),Starting_Pos,Race_Result
0,ALB,83.130,81.847,81.664,36.650031,20.898312,42.942393,100.398929,11,HARD,80.941,15.0,12.0
1,ANT,81.376,79.943,80.324,36.042935,19.788161,41.798556,95.665077,15,HARD,78.811,2.0,2.0
2,BEA,82.682,81.326,80.778,39.039839,20.835774,41.873786,102.072750,15,HARD,80.311,12.0,7.0
3,BOR,81.696,81.668,80.459,39.209964,20.777286,46.708920,99.549182,9,MEDIUM,80.221,10.0,9.0
4,BOT,84.022,83.660,83.514,37.285821,20.556821,43.604600,101.500360,15,MEDIUM,83.244,19.0,19.0
5,COL,83.325,82.619,81.413,38.297846,22.258222,45.171583,101.418318,11,MEDIUM,81.270,16.0,14.0
6,GAS,84.035,82.167,81.071,37.261133,22.742125,41.458429,99.050308,11,MEDIUM,80.501,14.0,10.0
7,HAD,81.087,80.941,80.137,36.938357,23.917321,42.749958,99.028217,13,MEDIUM,79.303,3.0,20.0
8,HAM,80.736,80.050,79.669,36.481677,21.753969,43.648655,99.920000,13,HARD,79.478,7.0,4.0
9,HUL,81.969,81.351,81.067,34.139485,20.528765,41.426839,93.962483,10,MEDIUM,80.303,11.0,22.0


In [15]:
df.to_csv("AustraliaGP.csv",index=False)

# Japan GP Data (For Training)

In [16]:
season = 2026
race = 'Japan'

FP1 = fastf1.get_session(season, race, 'FP1')
FP1.load()
FP2 = fastf1.get_session(season, race, 'FP2')
FP2.load()
FP3 = fastf1.get_session(season, race, 'FP3')
FP3.load()

Qualify = fastf1.get_session(season, race, 'Q')
Qualify.load()

Result = fastf1.get_session(season, race, 'R')
Result.load()

print(f"Done Loading Season {season} & Race {race}")

fp1_best = get_fp1_best(FP1)
fp2_best = get_fp2_best(FP2)
fp3_best = get_fp3_best(FP3)
sector_time_best = get_sector_times(FP2)
average_laptime = get_average_laptime(FP2)
stint = get_stint(FP2)
tyre = get_tyre_compound(FP2)
qualifying_time = get_qualifying_data(Qualify)
starting_position = get_starting_position(Qualify)
result = get_race_results(Result)

print("Data Extraction Done.")

fp1_best.set_index("Driver",inplace=True)
fp2_best.set_index("Driver",inplace=True)
fp3_best.set_index("Driver",inplace=True)
sector_time_best.set_index("Driver",inplace=True)
average_laptime.set_index("Driver",inplace=True)
stint.set_index("Driver",inplace=True)
tyre.set_index("Driver",inplace=True)
qualifying_time.set_index("Driver",inplace=True)
starting_position.set_index("Driver",inplace=True)
result.set_index("Driver",inplace=True)

print("Setting Index Done.")

df = fp1_best.copy()
df.rename(columns={"LapTime":"FP1_BestTime(s)"},inplace=True)
df[["FP2_BestTime(s)"]] = fp2_best[["LapTime"]]
df[["FP3_BestTime(s)"]] = fp3_best[["LapTime"]]
df[["Sector1Time(s)","Sector2Time(s)","Sector3Time(s)"]] = sector_time_best[["Sector1Time","Sector2Time","Sector3Time"]]
df[["Average_Laptime(s)"]] = average_laptime[["LapTime"]]
df[["Longest_Stint"]] = stint[["LongestStint"]]
df[["Tyre_Compound"]] = tyre[["Compound"]]
df[["Qualifying_Time(s)"]] = qualifying_time[["Qualifying_Time(s)"]]
df[["Starting_Pos"]] = starting_position[["Position"]]
df[["Race_Result"]] = result[["Position"]]

print("Data Transformation.")

fp1_best.reset_index(inplace=True)
fp2_best.reset_index(inplace=True)
fp3_best.reset_index(inplace=True)
sector_time_best.reset_index(inplace=True)
average_laptime.reset_index(inplace=True)
stint.reset_index(inplace=True)
tyre.reset_index(inplace=True)
qualifying_time.reset_index(inplace=True)
starting_position.reset_index(inplace=True)
result.reset_index(inplace=True)
df.reset_index(inplace=True)

print("Resetting Index Done.")


core           INFO 	Loading data for Japanese Grand Prix - Practice 1 [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '5', '6', '10', '11', '12', '16', '18', '23', '27', '30', '31', '34', '41', '43', '44', '55', '63', '77', '81', '87']
core           INFO 	Loading data for Japanese Grand Prix - Practice 2 [v3.8.1]
req            I

Done Loading Season 2026 & Race Japan
Data Extraction Done.
Setting Index Done.
Data Transformation.
Resetting Index Done.


/tmp/ipykernel_13829/2861812837.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  quali_results.rename(columns={"Abbreviation":"Driver"},inplace=True)
/tmp/ipykernel_13829/2097842272.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race.rename(columns={"Abbreviation":"Driver"},inplace=True)


In [17]:
df

,Driver,FP1_BestTime(s),FP2_BestTime(s),FP3_BestTime(s),Sector1Time(s),Sector2Time(s),Sector3Time(s),Average_Laptime(s),Longest_Stint,Tyre_Compound,Qualifying_Time(s),Starting_Pos,Race_Result
0,ALB,93.697,91.496,91.733,42.531231,48.057667,21.567000,106.666583,11.0,MEDIUM,91.088,17.0,20.0
1,ANT,91.692,90.225,89.362,47.109174,45.386107,21.655821,103.388474,10.0,MEDIUM,88.778,1.0,1.0
2,BEA,92.900,91.498,91.558,43.953800,45.669714,22.014214,102.402952,15.0,MEDIUM,91.090,18.0,22.0
3,BOR,92.759,91.933,91.000,53.260667,48.081545,24.916545,119.585250,7.0,SOFT,90.274,9.0,13.0
4,BOT,94.490,92.615,92.503,43.989200,47.002679,21.424036,109.386375,15.0,MEDIUM,92.330,20.0,19.0
5,COL,93.361,92.438,91.759,43.006280,47.661536,21.347107,104.757091,14.0,MEDIUM,90.627,15.0,16.0
6,CRA,96.362,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,GAS,92.978,91.734,91.082,41.008560,46.617310,21.627517,106.528667,13.0,HARD,89.691,7.0,7.0
8,HAD,92.803,91.759,91.094,45.154038,46.109931,21.571241,106.243957,14.0,HARD,89.978,8.0,12.0
9,HAM,92.040,90.980,90.383,46.753333,49.164185,22.050000,104.465000,9.0,MEDIUM,89.567,6.0,6.0


In [18]:
df.to_csv("JapanGP.csv",index=False)

# China GP Data (For Training)

In [19]:
season = 2026
race = 'China'

FP1 = fastf1.get_session(season, race, 'FP1')
FP1.load()
FP2 = fastf1.get_session(season, race, 'SQ')
FP2.load()
FP3 = fastf1.get_session(season, race, 'S')
FP3.load()

Qualify = fastf1.get_session(season, race, 'Q')
Qualify.load()

Result = fastf1.get_session(season, race, 'R')
Result.load()

print(f"Done Loading Season {season} & Race {race}")

fp1_best = get_fp1_best(FP1)
fp2_best = get_fp2_best(FP2)
fp3_best = get_fp3_best(FP3)
sector_time_best = get_sector_times(FP2)
average_laptime = get_average_laptime(FP2)
stint = get_stint(FP2)
tyre = get_tyre_compound(FP2)
qualifying_time = get_qualifying_data(Qualify)
starting_position = get_starting_position(Qualify)
result = get_race_results(Result)

print("Data Extraction Done.")

fp1_best.set_index("Driver",inplace=True)
fp2_best.set_index("Driver",inplace=True)
fp3_best.set_index("Driver",inplace=True)
sector_time_best.set_index("Driver",inplace=True)
average_laptime.set_index("Driver",inplace=True)
stint.set_index("Driver",inplace=True)
tyre.set_index("Driver",inplace=True)
qualifying_time.set_index("Driver",inplace=True)
starting_position.set_index("Driver",inplace=True)
result.set_index("Driver",inplace=True)

print("Setting Index Done.")

df = fp1_best.copy()
df.rename(columns={"LapTime":"FP1_BestTime(s)"},inplace=True)
df[["FP2_BestTime(s)"]] = fp2_best[["LapTime"]]
df[["FP3_BestTime(s)"]] = fp3_best[["LapTime"]]
df[["Sector1Time(s)","Sector2Time(s)","Sector3Time(s)"]] = sector_time_best[["Sector1Time","Sector2Time","Sector3Time"]]
df[["Average_Laptime(s)"]] = average_laptime[["LapTime"]]
df[["Longest_Stint"]] = stint[["LongestStint"]]
df[["Tyre_Compound"]] = tyre[["Compound"]]
df[["Qualifying_Time(s)"]] = qualifying_time[["Qualifying_Time(s)"]]
df[["Starting_Pos"]] = starting_position[["Position"]]
df[["Race_Result"]] = result[["Position"]]

print("Data Transformation.")

fp1_best.reset_index(inplace=True)
fp2_best.reset_index(inplace=True)
fp3_best.reset_index(inplace=True)
sector_time_best.reset_index(inplace=True)
average_laptime.reset_index(inplace=True)
stint.reset_index(inplace=True)
tyre.reset_index(inplace=True)
qualifying_time.reset_index(inplace=True)
starting_position.reset_index(inplace=True)
result.reset_index(inplace=True)
df.reset_index(inplace=True)

print("Resetting Index Done.")


core           INFO 	Loading data for Chinese Grand Prix - Practice 1 [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '5', '6', '10', '11', '12', '14', '16', '18', '23', '27', '30', '31', '41', '43', '44', '55', '63', '77', '81', '87']
core           INFO 	Loading data for Chinese Grand Prix - Sprint Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core        WARNING 	Sprint Qualifying is not supported by Ergast! Limited results are cal

Done Loading Season 2026 & Race China
Data Extraction Done.
Setting Index Done.
Data Transformation.
Resetting Index Done.


/tmp/ipykernel_13829/2861812837.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  quali_results.rename(columns={"Abbreviation":"Driver"},inplace=True)
/tmp/ipykernel_13829/2097842272.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race.rename(columns={"Abbreviation":"Driver"},inplace=True)


In [20]:
df

,Driver,FP1_BestTime(s),FP2_BestTime(s),FP3_BestTime(s),Sector1Time(s),Sector2Time(s),Sector3Time(s),Average_Laptime(s),Longest_Stint,Tyre_Compound,Qualifying_Time(s),Starting_Pos,Race_Result
0,ALB,95.480,95.305,98.158,27.300600,31.527667,49.645000,108.962200,6.0,MEDIUM,94.772,18.0,22.0
1,ALO,95.856,95.581,98.487,28.490000,31.746600,48.075000,107.564250,5.0,MEDIUM,95.203,19.0,17.0
2,ANT,92.861,91.809,94.760,28.116400,32.863769,47.599231,107.658700,5.0,MEDIUM,92.064,1.0,1.0
3,BEA,94.426,93.409,97.374,29.538154,31.894063,48.543625,110.244462,7.0,MEDIUM,93.292,10.0,5.0
4,BOR,94.828,93.774,97.800,28.943000,32.736714,48.478214,110.359250,8.0,MEDIUM,93.965,16.0,21.0
5,BOT,96.057,97.378,99.262,28.484571,33.107250,48.428500,109.943714,8.0,MEDIUM,95.436,20.0,13.0
6,COL,94.947,94.327,97.278,30.291600,33.194917,47.793000,111.528000,6.0,MEDIUM,93.357,12.0,10.0
7,GAS,94.676,92.888,98.349,29.145417,32.768933,49.672133,112.275500,6.0,MEDIUM,92.873,7.0,6.0
8,HAD,94.856,93.620,96.912,30.018583,31.808333,47.518714,107.512636,7.0,MEDIUM,93.121,9.0,8.0
9,HAM,94.129,92.161,94.926,28.891417,33.737000,49.144600,111.767250,5.0,MEDIUM,92.415,3.0,3.0


In [21]:
df.to_csv("ChinaGP.csv",index=False)

# Miami GP (For Prediction)

In [22]:
season = 2026
race = 'Miami'

FP1 = fastf1.get_session(season, race, 'FP1')
FP1.load()
FP2 = fastf1.get_session(season, race, 'SQ')
FP2.load()
FP3 = fastf1.get_session(season, race, 'S')
FP3.load()

Qualify = fastf1.get_session(season, race, 'Q')
Qualify.load()


print(f"Done Loading Season {season} & Race {race}")

fp1_best = get_fp1_best(FP1)
fp2_best = get_fp2_best(FP2)
fp3_best = get_fp3_best(FP3)
sector_time_best = get_sector_times(FP2)
average_laptime = get_average_laptime(FP2)
stint = get_stint(FP2)
tyre = get_tyre_compound(FP2)
qualifying_time = get_qualifying_data(Qualify)
starting_position = get_starting_position(Qualify)

print("Data Extraction Done.")

fp1_best.set_index("Driver",inplace=True)
fp2_best.set_index("Driver",inplace=True)
fp3_best.set_index("Driver",inplace=True)
sector_time_best.set_index("Driver",inplace=True)
average_laptime.set_index("Driver",inplace=True)
stint.set_index("Driver",inplace=True)
tyre.set_index("Driver",inplace=True)
qualifying_time.set_index("Driver",inplace=True)
starting_position.set_index("Driver",inplace=True)
print("Setting Index Done.")

df = fp1_best.copy()
df.rename(columns={"LapTime":"FP1_BestTime(s)"},inplace=True)
df[["FP2_BestTime(s)"]] = fp2_best[["LapTime"]]
df[["FP3_BestTime(s)"]] = fp3_best[["LapTime"]]
df[["Sector1Time(s)","Sector2Time(s)","Sector3Time(s)"]] = sector_time_best[["Sector1Time","Sector2Time","Sector3Time"]]
df[["Average_Laptime(s)"]] = average_laptime[["LapTime"]]
df[["Longest_Stint"]] = stint[["LongestStint"]]
df[["Tyre_Compound"]] = tyre[["Compound"]]
df[["Qualifying_Time(s)"]] = qualifying_time[["Qualifying_Time(s)"]]
df[["Starting_Pos"]] = starting_position[["Position"]]

print("Data Transformation.")

fp1_best.reset_index(inplace=True)
fp2_best.reset_index(inplace=True)
fp3_best.reset_index(inplace=True)
sector_time_best.reset_index(inplace=True)
average_laptime.reset_index(inplace=True)
stint.reset_index(inplace=True)
tyre.reset_index(inplace=True)
qualifying_time.reset_index(inplace=True)
starting_position.reset_index(inplace=True)
df.reset_index(inplace=True)

print("Resetting Index Done.")


core           INFO 	Loading data for Miami Grand Prix - Practice 1 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data..

Done Loading Season 2026 & Race Miami
Data Extraction Done.
Setting Index Done.
Data Transformation.
Resetting Index Done.


/tmp/ipykernel_13829/2861812837.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  quali_results.rename(columns={"Abbreviation":"Driver"},inplace=True)


In [23]:
df

,Driver,FP1_BestTime(s),FP2_BestTime(s),FP3_BestTime(s),Sector1Time(s),Sector2Time(s),Sector3Time(s),Average_Laptime(s),Longest_Stint,Tyre_Compound,Qualifying_Time(s),Starting_Pos
0,ALB,91.024,90.216,93.473,40.930200,40.687333,31.818667,112.258500,6,MEDIUM,89.946,16.0
1,ALO,92.593,92.490,94.720,37.105200,39.793167,34.718667,111.428600,6,MEDIUM,91.098,18.0
2,ANT,90.079,88.091,91.932,39.886700,40.724857,33.187923,106.295500,3,MEDIUM,87.798,1.0
3,BEA,91.091,90.116,94.524,40.357000,39.887444,31.671333,110.144833,3,MEDIUM,89.567,13.0
4,BOR,91.111,89.994,94.174,37.460000,41.726500,33.908167,111.922889,6,MEDIUM,93.737,22.0
5,BOT,92.762,91.826,95.856,36.451500,40.314667,36.732800,92.071500,3,MEDIUM,91.629,20.0
6,COL,91.015,89.320,93.139,39.173833,39.762400,31.374500,109.832000,6,MEDIUM,88.762,8.0
7,GAS,90.587,89.474,93.072,39.254417,40.081667,30.743786,109.097636,6,MEDIUM,88.810,10.0
8,HAD,90.873,89.422,93.119,36.692083,40.412400,31.293143,106.426636,6,MEDIUM,88.789,9.0
9,HAM,89.777,88.618,92.448,38.031750,39.298200,30.978429,107.213273,6,MEDIUM,88.319,6.0


In [24]:
df.to_csv("MiamiGP.csv",index=False)